In [1]:
%matplotlib widget
import pandas as pd
import pickle
from pyrfu import pyrf
import xarray as xr
import numpy as np
from pyrfu import mms
from matplotlib import pyplot as plt
mms.db_init(default="local", local="/data/mms")


Load IGRF coefficients ...


[07-Feb-26 17:02:22] INFO: Updating MMS data access configuration in /homelocal/apostolosk/envs/mms_projects/lib/python3.12/site-packages/pyrfu/mms/config.json...
[07-Feb-26 17:02:22] INFO: Updating MMS SDC credentials in /homelocal/apostolosk/.config/python_keyring...


In [2]:
bbfs_df = xr.load_dataset("mms_bbfsdb_2017-2021.nc")
solitary_dfs = bbfs_df.where(bbfs_df["is_df"] == 1, drop = True)

t_0 = np.datetime64("2014-01-01T00:00:00")
t_lim = np.datetime64("2018-06-11T00:00:00")

solitary_dfs = solitary_dfs.sel(time  = slice(t_0, t_lim))

In [3]:
id_orig = []
tints = []
dt = np.timedelta64(30, "s")
for i in range(len(solitary_dfs.t_df)):
    tints.append([str(solitary_dfs.t_df[i-1].data - dt), str(solitary_dfs.t_df[i-1].data + dt)])
    

In [4]:
tints_no_brst = []
tints_brst = []
t_center_brst = []
for i in range(len(tints)):

    try:
       
        ve = [mms.get_data("ve_gse_fpi_brst_l2", tints[i], mms_id=id) for id in range(1, 5)]

        count_empty = False
        for mms_i in range(4):
            if len(ve[mms_i].time) == 0:
                count_empty = True
                break
        if count_empty:
            tints_no_brst.append(tints[i])
        else:
            id_orig.append(i)
            tints_brst.append(tints[i])
            t_center_brst.append(solitary_dfs.t_df[i].data)
    except:
        tints_no_brst.append(tints[i]) 

[07-Feb-26 17:02:34] INFO: Loading mms1_des_bulkv_gse_brst...
[07-Feb-26 17:02:34] INFO: Loading mms2_des_bulkv_gse_brst...
[07-Feb-26 17:02:35] INFO: Loading mms3_des_bulkv_gse_brst...
[07-Feb-26 17:02:35] INFO: Loading mms4_des_bulkv_gse_brst...
[07-Feb-26 17:02:35] INFO: Loading mms1_des_bulkv_gse_brst...
[07-Feb-26 17:02:35] INFO: Loading mms2_des_bulkv_gse_brst...
[07-Feb-26 17:02:35] INFO: Loading mms3_des_bulkv_gse_brst...
[07-Feb-26 17:02:36] INFO: Loading mms4_des_bulkv_gse_brst...
[07-Feb-26 17:02:36] INFO: Loading mms1_des_bulkv_gse_brst...
[07-Feb-26 17:02:36] INFO: Loading mms2_des_bulkv_gse_brst...
[07-Feb-26 17:02:36] INFO: Loading mms3_des_bulkv_gse_brst...
[07-Feb-26 17:02:36] INFO: Loading mms4_des_bulkv_gse_brst...
[07-Feb-26 17:02:36] INFO: Loading mms1_des_bulkv_gse_brst...
[07-Feb-26 17:02:36] INFO: Loading mms2_des_bulkv_gse_brst...
[07-Feb-26 17:02:37] INFO: Loading mms3_des_bulkv_gse_brst...
[07-Feb-26 17:02:37] INFO: Loading mms4_des_bulkv_gse_brst...
[07-Feb-

In [5]:
sol_dfs_brst = solitary_dfs.isel(time = id_orig)

In [6]:
pd.DataFrame(tints_brst).to_csv("tints_dfs_brst.csv")